In [120]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
from catboost import CatBoostRegressor

In [121]:
SEED     = 42
N_SPLITS = 2
HORIZON  = 14
DATA_DIR = Path(".")
OUT_PATH = DATA_DIR / "submission1.csv"

np.random.seed(SEED)

In [122]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv",  parse_dates=["date"])
sub = pd.read_csv(DATA_DIR / "ans.csv")

train["_split"] = "train"
test["_split"]  = "test"
test["demand"]  = np.nan

df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["store_id", "product_id", "date"]).reset_index(drop=True)


In [123]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["date"].dt.quarter
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["woy_sin"] = np.sin(2 * np.pi * df["weekofyear"] / 52)
df["woy_cos"] = np.cos(2 * np.pi * df["weekofyear"] / 52)


In [124]:
holiday_dates = df.loc[df["is_holiday"] == 1, "date"].unique()

def days_to_nearest_holiday(d):
    if len(holiday_dates) == 0:
        return 0
    return float(np.abs((holiday_dates - d) / np.timedelta64(1, "D")).min())

hd_sorted = np.sort(holiday_dates)

def days_since_last(d):
    past = hd_sorted[hd_sorted <= d]
    return float((d - past[-1]) / np.timedelta64(1, "D")) if len(past) else 999.0

def days_until_next(d):
    future = hd_sorted[hd_sorted > d]
    return float((future[0] - d) / np.timedelta64(1, "D")) if len(future) else 999.0

unique_dates = df["date"].unique()
df["days_to_holiday"]    = df["date"].map({d: days_to_nearest_holiday(d) for d in unique_dates})
df["days_since_holiday"] = df["date"].map({d: days_since_last(d) for d in unique_dates})
df["days_until_holiday"] = df["date"].map({d: days_until_next(d) for d in unique_dates})


In [125]:
grp = df.groupby(["store_id", "product_id"])["demand"]

# Short-term + weekly lags
for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]:
    df[f"lag_{lag}"] = grp.shift(lag)

# Same day-of-week across 4 past weeks
for lag in [7, 14, 21, 28]:
    df[f"lag_same_dow_{lag}d"] = grp.shift(lag)


In [126]:
def add_rolling(df, window, shift=7):
    shifted = df.groupby(["store_id", "product_id"])["demand"].shift(shift)
    rolled  = shifted.groupby([df["store_id"], df["product_id"]]) \
                     .rolling(window, min_periods=1)
    df[f"roll_mean_{window}"] = rolled.mean().droplevel([0, 1])
    df[f"roll_std_{window}"]  = rolled.std().droplevel([0, 1])
    df[f"roll_max_{window}"]  = rolled.max().droplevel([0, 1])
    df[f"roll_min_{window}"]  = rolled.min().droplevel([0, 1])
    return df

for window in [7, 14, 28, 56, 90]:
    df = add_rolling(df, window)

df["sell_through_trend"]  = df["roll_mean_7"]  / (df["roll_mean_28"] + 1e-6)
df["sell_through_trend2"] = df["roll_mean_14"] / (df["roll_mean_90"] + 1e-6)

df["expanding_mean"] = (
    df.groupby(["store_id", "product_id"])["demand"]
      .shift(1)
      .groupby([df["store_id"], df["product_id"]])
      .transform(lambda x: x.expanding().mean())
)


In [127]:
def ewm_feature(df, span, shift=1):
    return (
        df.groupby(["store_id", "product_id"])["demand"]
          .shift(shift)
          .groupby([df["store_id"], df["product_id"]])
          .transform(lambda x: x.ewm(span=span, min_periods=1).mean())
    )

for span in [7, 14, 28]:
    df[f"ewm_mean_{span}"] = ewm_feature(df, span)

df["ewm_trend"] = df["ewm_mean_7"] / (df["ewm_mean_28"] + 1e-6)


In [128]:
store_prod_median = (
    train.groupby(["store_id", "product_id"])["price"].median()
         .rename("price_store_median")
)
df = df.join(store_prod_median, on=["store_id", "product_id"])
df["price_rel_store"] = df["price"] / (df["price_store_median"] + 1e-6)

city_prod_median = (
    train.groupby(["city", "product_id"])["price"].median()
         .rename("price_city_median")
)
df = df.join(city_prod_median, on=["city", "product_id"])
df["price_rel_city"] = df["price"] / (df["price_city_median"] + 1e-6)

df["price_lag1"] = df.groupby(["store_id", "product_id"])["price"].shift(1)
df["price_delta"] = df["price"] - df["price_lag1"]
df["price_pct_chg"] = df["price_delta"] / (df["price_lag1"] + 1e-6)

df["price_rank_in_category"] = (
    df.groupby(["date", "category"])["price"].rank(pct=True)
)

df["demand_per_price"] = df["roll_mean_7"] / (df["price"] + 1e-6)



In [129]:
store_stats = (
    train.groupby("store_id")["demand"]
         .agg(store_demand_mean="mean", store_demand_std="std")
)
df = df.join(store_stats, on="store_id")

city_prod_rank = (
    train.groupby(["city", "product_id"])["demand"].mean()
         .groupby(level=0).rank(ascending=False)
         .rename("city_prod_rank")
)
df = df.join(city_prod_rank, on=["city", "product_id"])

region_mean = (
    train.groupby("region")["demand"].mean().rename("region_demand_mean")
)
df = df.join(region_mean, on="region")

n_competitors = (
    train.groupby(["city", "product_id"])["store_id"].nunique()
         .sub(1).rename("n_competitors")
)
df = df.join(n_competitors, on=["city", "product_id"])
df["n_competitors"] = df["n_competitors"].fillna(0)



In [130]:

df["loyalty_x_weekend"] = df["loyalty_day"] * df["is_weekend"]
df["loyalty_x_dow"]     = df["loyalty_day"] * df["dayofweek"]
df["loyalty_x_holiday"] = df["loyalty_day"] * df["is_holiday"]

loyalty_mean = (
    train[train["loyalty_day"] == 1]
         .groupby(["store_id", "product_id"])["demand"].mean()
         .rename("loyalty_demand_mean")
)
normal_mean = (
    train[train["loyalty_day"] == 0]
         .groupby(["store_id", "product_id"])["demand"].mean()
         .rename("normal_demand_mean")
)
df = df.join(loyalty_mean, on=["store_id", "product_id"])
df = df.join(normal_mean,  on=["store_id", "product_id"])
df["loyalty_lift"] = df["loyalty_demand_mean"] / (df["normal_demand_mean"] + 1e-6)


In [131]:

first_seen = (
    train.groupby(["store_id", "product_id"])["date"].min()
         .rename("first_seen_date")
)
df = df.join(first_seen, on=["store_id", "product_id"])
df["product_age_days"] = (df["date"] - df["first_seen_date"]).dt.days.clip(lower=0)
df["is_new_product"]   = (df["product_age_days"] < 30).astype(int)


In [132]:
cat_cols = ["store_id", "product_id", "city", "region",
            "product_name", "category", "holiday_name"]

for col in cat_cols:
    df[col] = df[col].astype(str).fillna("MISSING")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])


In [133]:
tr = df[df["_split"] == "train"].copy()
te = df[df["_split"] == "test"].copy()

tr["week_group"] = (tr["date"] - tr["date"].min()).dt.days // 7
y_train = tr["demand"].clip(lower=0)
GLOBAL_MEAN = float(y_train.mean())

TE_CONFIGS = [
    ["store_id", "product_id"],
    ["category", "dayofweek"],
    ["city", "product_id"],
    ["region", "category"],
]
TE_COL_NAMES = [f"te_{'_'.join(cols)}" for cols in TE_CONFIGS]

DROP_COLS = [
    "row_ID", "date", "demand", "_split", "week_group",
    "price_lag1", "first_seen_date",
    "price_store_median", "price_city_median",
    "loyalty_demand_mean", "normal_demand_mean",
]

BASE_FEATURES = [c for c in df.columns if c not in DROP_COLS + TE_COL_NAMES]
ALL_FEATURES  = BASE_FEATURES + TE_COL_NAMES

In [134]:
def target_encode(train_df, apply_df, key_cols, smoothing=20):

    col_name = f"te_{'_'.join(key_cols)}"
    agg = (
        train_df.groupby(key_cols)["demand"]
                .agg(["mean", "count"])
                .reset_index()
    )
    agg[col_name] = (
        (agg["mean"] * agg["count"] + GLOBAL_MEAN * smoothing)
        / (agg["count"] + smoothing)
    )
    result = (
        apply_df[key_cols]
               .merge(agg[key_cols + [col_name]], on=key_cols, how="left")[col_name]
               .fillna(GLOBAL_MEAN)
    )
    result.index = apply_df.index
    return result


In [135]:
import tqdm

gkf = GroupKFold(n_splits=N_SPLITS)
cv_groups = tr["week_group"]

for col in TE_COL_NAMES:
    te[col] = 0.0

cb_params = {
    "loss_function": "RMSE",
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "subsample": 0.8,
    "colsample_bylevel": 0.8,
    "min_data_in_leaf": 30,
    "random_seed": SEED,
    "thread_count": -1,
    "verbose": 500,
    "early_stopping_rounds": 100,
}

oof_cb  = np.zeros(len(tr))
pred_cb = np.zeros(len(te))

for fold, (idx_tr, idx_val) in tqdm.tqdm(enumerate(
        gkf.split(tr[BASE_FEATURES], y_train, groups=cv_groups), 1)):

    tr_fold  = tr.iloc[idx_tr].copy()
    val_fold = tr.iloc[idx_val].copy()
    ytr  = y_train.iloc[idx_tr]
    yval = y_train.iloc[idx_val]

    te_fold = te.copy()

    for key_cols in tqdm.tqdm(TE_CONFIGS):
        col = f"te_{'_'.join(key_cols)}"
        tr_fold[col] = target_encode(tr_fold, tr_fold,  key_cols)
        val_fold[col] = target_encode(tr_fold, val_fold, key_cols)
        te_fold[col] = target_encode(tr_fold, te_fold,  key_cols)
        te[col] += te_fold[col] / N_SPLITS

    model = CatBoostRegressor(**cb_params)
    model.fit(
        tr_fold[ALL_FEATURES], ytr,
        eval_set=(val_fold[ALL_FEATURES], yval),
    )

    oof_cb[idx_val] = model.predict(val_fold[ALL_FEATURES]).clip(min=0)
    pred_cb += model.predict(te_fold[ALL_FEATURES]).clip(min=0) / N_SPLITS

    print(f"  Fold {fold} mae: {mean_absolute_error(yval, oof_cb[idx_val]):.4f}"
          f" | best iter: {model.best_iteration_}")

cb_oof_mae = mean_absolute_error(y_train, oof_cb)
print(f"\n  cb oof mae: {cb_oof_mae:.4f}")


0it [00:00, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 24.88it/s]


0:	learn: 2.6496671	test: 2.7552915	best: 2.7552915 (0)	total: 27.1ms	remaining: 27.1s
500:	learn: 1.7103155	test: 1.9006782	best: 1.9005493 (493)	total: 8.71s	remaining: 8.67s


1it [00:17, 17.53s/it]

999:	learn: 1.6407769	test: 1.8965235	best: 1.8965235 (999)	total: 16.7s	remaining: 0us

bestTest = 1.896523539
bestIteration = 999

  Fold 1 mae: 1.0321 | best iter: 999


100%|██████████| 4/4 [00:00<00:00, 25.17it/s]


0:	learn: 2.7522702	test: 2.6473774	best: 2.6473774 (0)	total: 18.7ms	remaining: 18.7s
500:	learn: 1.7486432	test: 1.8464268	best: 1.8464115 (499)	total: 7.94s	remaining: 7.91s


2it [00:30, 15.15s/it]

Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.844077783
bestIteration = 665

Shrink model to first 666 iterations.
  Fold 2 mae: 1.0251 | best iter: 665

  cb oof mae: 1.0286


In [136]:

xgb_params = {
    "objective": "reg:absoluteerror",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 30,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.2,
    "reg_lambda": 0.2,
    "random_state": SEED,
    "n_jobs": -1,
    "tree_method": "hist",
    "verbosity": 0,
}

oof_xgb  = np.zeros(len(tr))
pred_xgb = np.zeros(len(te))

for fold, (idx_tr, idx_val) in enumerate(
        gkf.split(tr[BASE_FEATURES], y_train, groups=cv_groups), 1):

    tr_fold  = tr.iloc[idx_tr].copy()
    val_fold = tr.iloc[idx_val].copy()
    ytr  = y_train.iloc[idx_tr]
    yval = y_train.iloc[idx_val]

    te_fold = te.copy()

    for key_cols in TE_CONFIGS:
        col = f"te_{'_'.join(key_cols)}"
        tr_fold[col] = target_encode(tr_fold, tr_fold,  key_cols)
        val_fold[col] = target_encode(tr_fold, val_fold, key_cols)
        te_fold[col] = target_encode(tr_fold, te_fold,  key_cols)

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        tr_fold[ALL_FEATURES], ytr,
        eval_set=[(val_fold[ALL_FEATURES], yval)],
        verbose=500,
    )

    oof_xgb[idx_val] = model.predict(val_fold[ALL_FEATURES]).clip(min=0)
    pred_xgb += model.predict(te_fold[ALL_FEATURES]).clip(min=0) / N_SPLITS

    print(f"  Fold {fold} mae: {mean_absolute_error(yval, oof_xgb[idx_val]):.4f}")

xgb_oof_mae = mean_absolute_error(y_train, oof_xgb)
print(f"\n  xg oof mae: {xgb_oof_mae:.4f}")

[0]	validation_0-mae:1.36296
[500]	validation_0-mae:0.95888
[999]	validation_0-mae:0.95649
  Fold 1 mae: 0.9556
[0]	validation_0-mae:1.32648
[500]	validation_0-mae:1.00473
[999]	validation_0-mae:1.00704
  Fold 2 mae: 1.0064

  xg oof mae: 0.9810


In [137]:

w_cb  = 1 / cb_oof_mae
w_xgb = 1 / xgb_oof_mae
w_sum = w_cb + w_xgb

base_preds = (w_cb * pred_cb + w_xgb * pred_xgb) / w_sum

oof_ensemble = (w_cb * oof_cb  + w_xgb * oof_xgb)  / w_sum
ens_oof_mae = mean_absolute_error(y_train, oof_ensemble)
print(f"ens oof mae : {ens_oof_mae:.4f}")


print(f"cb weight: {w_cb/w_sum:.3f}  |  xgb weight: {w_xgb/w_sum:.3f}")


ens oof mae : 0.9938
cb weight: 0.488  |  xgb weight: 0.512


In [138]:
tr_final = tr.copy()
te_final = te.copy()

for key_cols in TE_CONFIGS:
    col = f"te_{'_'.join(key_cols)}"
    tr_final[col] = target_encode(tr_final, tr_final, key_cols)

final_cb = CatBoostRegressor(**{**cb_params,
                                 "iterations": 2000,
                                 "early_stopping_rounds": None})

final_cb.fit(tr_final[ALL_FEATURES], y_train)

final_xgb = xgb.XGBRegressor(**{**xgb_params, "n_estimators": 2000})
final_xgb.fit(tr_final[ALL_FEATURES], y_train)

last_train_rows = (
    train.sort_values("date")
         .groupby(["store_id", "product_id"])
         .tail(HORIZON * 3)
         [["store_id", "product_id", "date", "demand"]]
)

demand_lookup = {}

for row in last_train_rows.itertuples(index=False):
    demand_lookup[(row.store_id, row.product_id, row.date)] = row.demand

test_dates = sorted(te_final["date"].unique())
recursive_preds = {}


0:	learn: 2.7015102	total: 51.3ms	remaining: 1m 42s
500:	learn: 1.7685668	total: 15.5s	remaining: 46.4s
1000:	learn: 1.7168929	total: 30.6s	remaining: 30.6s
1500:	learn: 1.6776447	total: 45.7s	remaining: 15.2s
1999:	learn: 1.6468436	total: 1m 1s	remaining: 0us


In [139]:

for day_offset, pred_date in enumerate(test_dates):
    day_df = te_final[te_final["date"] == pred_date].copy()

    for lag in [1, 2, 3, 4, 5, 6, 7, 14]:
        lag_col  = f"lag_{lag}"
        lag_date = pred_date - pd.Timedelta(days=lag)
        if lag_col not in day_df.columns:
            continue
        day_df[lag_col] = day_df.apply(
            lambda r, ld=lag_date, lc=lag_col: demand_lookup.get(
                (r["store_id"], r["product_id"], ld), r[lc]
            ), axis=1
        )
    cb_p = final_cb.predict(day_df[ALL_FEATURES]).clip(min=0)
    xgb_p = final_xgb.predict(day_df[ALL_FEATURES]).clip(min=0)
    day_pred = (w_cb * cb_p + w_xgb * xgb_p) / w_sum

    for i, row_id in enumerate(day_df["row_id"].values):
        sp = day_df["store_id"].iloc[i]
        pp = day_df["product_id"].iloc[i]
        demand_lookup[(sp, pp, pred_date)] = day_pred[i]
        recursive_preds[row_id] = day_pred[i]

    print(f"Day {day_offset+1:2d}/{HORIZON}  ({pred_date.date()})  "
          f"mean_pred={day_pred.mean():.2f}")

te_final["recursive_pred"] = te_final["row_id"].map(recursive_preds).clip(lower=0)

final_preds = (0.5 * base_preds + 0.5 * te_final["recursive_pred"].values).clip(min=0)


te_result = te_final[["row_id"]].copy()
te_result["demand"] = final_preds

sub_out = sub[["row_id"]].merge(te_result, on="row_id", how="left")
sub_out["demand"] = sub_out["demand"].fillna(0).clip(lower=0)

sub_out.to_csv(OUT_PATH, index=False)
display(sub_out.head(10))


Day  1/14  (2025-05-11)  mean_pred=1.50
Day  2/14  (2025-05-12)  mean_pred=1.37
Day  3/14  (2025-05-13)  mean_pred=1.39
Day  4/14  (2025-05-14)  mean_pred=1.50
Day  5/14  (2025-05-15)  mean_pred=1.68
Day  6/14  (2025-05-16)  mean_pred=1.90
Day  7/14  (2025-05-17)  mean_pred=1.82
Day  8/14  (2025-05-18)  mean_pred=1.47
Day  9/14  (2025-05-19)  mean_pred=1.40
Day 10/14  (2025-05-20)  mean_pred=1.38
Day 11/14  (2025-05-21)  mean_pred=1.48
Day 12/14  (2025-05-22)  mean_pred=1.73
Day 13/14  (2025-05-23)  mean_pred=1.94
Day 14/14  (2025-05-24)  mean_pred=1.84


,row_id,demand
0,0,6.207178
1,1,3.004303
2,2,1.248152
3,3,0.349823
4,4,0.441048
5,5,0.203364
6,6,0.342907
7,7,2.332216
8,8,1.734699
9,9,0.275724
